In [ ]:
import json
import time
import re
from pathlib import Path

In [ ]:
# Import kaggle_benchmarks as kbench for using free AI quota provided by Kaggle (10 usd/day - 100 usd/month). 
# This is a temporary solution to avoid the cost of using API as OpenAI.
import kaggle_benchmarks as kbench

# My project uses the "google/gemini-3.6-flash" provided by Kaggle Benchmarks.
llm=kbench.llms["google/gemini-3.6-flash"]

#  You can check the available LLMs by printing the keys of kbench.llms dictionary.
print(list(kbench.llms.keys())) 

In [ ]:
# Call the LLM many times can lead to exceeding the free quota or overloading the LLM,
# so you should consider splitting the files into parts and adding a delay between calls.

PART = 1
TOTAL_PARTS = 1
DELAY_BETWEEN_CALLS = 60.0

BATCH_SIZE = 10
MAX_RETRIES = 3

# The topic of the video, which will be used in the prompt to guide the editing process.
VIDEO_TOPIC = ""

def get_total_files(transcript_dir, part=PART, total_parts=TOTAL_PARTS):
    raw_paths = transcript_dir.glob("*_transcript.json")
    all_json_paths = sorted(set([str(p) for p in raw_paths]))
    total_all_files = len(all_json_paths)

    if total_all_files == 0:
        print(f"⚠️ Không tìm thấy file JSON nào trong {transcript_dir}")

    avg_chunk = total_all_files // total_parts
    start_idx = (part - 1) * avg_chunk
    end_idx = total_all_files if part == total_parts else part * avg_chunk

    part_paths = all_json_paths[start_idx:end_idx]

    print(f"🔥 PART {part}/{total_parts} - Transcript dir: {transcript_dir}")
    print(f"📊 Xử lý từ file #{start_idx + 1} đến #{end_idx} (Tổng {len(part_paths)}/{total_all_files} files)")
    print("----------------------------------------------------------------------")
    return part_paths


def build_prompt(texts):
    payload = "\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))
    return f"""Bạn là chuyên gia biên tập cho chương trình {VIDEO_TOPIC}. Hãy biên tập lại các đoạn (event) được đánh số dưới đây.
YÊU CẦU:
- Thêm dấu câu (chấm, phẩy, ...) và viết hoa đầu câu cho đúng.
- Sửa lỗi chính tả, lỗi Telex/VNI và các từ nghe nhầm âm khi ngữ cảnh đủ rõ (ví dụ địa danh, từ hành chính như "huyện", "Ủy ban", "sạt lở").
- Loại bỏ cụm từ lặp vô nghĩa (ví dụ "PHÔNG PHÔNG PHÔNG...").
- TUYỆT ĐỐI KHÔNG sửa tên riêng nước ngoài thành từ Việt hóa như "Renan Gôm", KHÔNG thêm hay bớt thông tin, KHÔNG tóm tắt, KHÔNG đổi số liệu.
- BẮT BUỘC trả về đúng {len(texts)} đoạn, mỗi đoạn bắt đầu bằng đúng số thứ tự dạng [1], [2], ... và nằm trên MỘT dòng. KHÔNG ghi chú, KHÔNG giải thích, KHÔNG code fence.

Danh sách các đoạn:
{payload}"""


def parse_response(response_text, expected):
    text = response_text.strip()
    text = re.sub(r"^```\w*\n?|\n?```$", "", text).strip()
    parts = re.split(r"(?m)^\s*\[(\d+)\]\s*", text)
    result = {}
    for k in range(1, len(parts) - 1, 2):
        result[int(parts[k])] = parts[k + 1].strip()
    if sorted(result.keys()) != list(range(1, expected + 1)):
        raise ValueError(f"Parse được {len(result)} đoạn, cần {expected}.")
    if any(not v for v in result.values()):
        raise ValueError("Có đoạn rỗng.")
    return [" ".join(result[i].split()) for i in range(1, expected + 1)]

In [ ]:
transcript_dir = Path("")  #folder containing all transcript files
base_output_dir = Path("") #folder to save the output files

@kbench.task(name=f"Correct Event Text", description=f"Thêm dấu câu và sửa lỗi chính tả cho text của các events")
def correct_event_text_task(llm) -> None:
    part_paths = get_total_files(transcript_dir)

    for idx, input_path in enumerate(part_paths, 1):
        filename = Path(input_path).name
        output_path = base_output_dir / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"🎬 [{idx}/{len(part_paths)}] Đang xử lý: {filename}")

        if output_path.resolve() == Path(input_path).resolve():
            raise RuntimeError(f"output_path trùng input_path ({input_path})!")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        video_key = next(iter(data), None)
        if video_key is None:
            print("    ⏩ File rỗng. Bỏ qua.")
            continue
            
        info = data[video_key]
        if info.get("status") != "success":
            print(f"    ⏩ Bỏ qua: status group = {info.get('status')}")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({video_key: {
                    "correct_status": "skipped", 
                    "group_status": info.get("status", "unknown"), 
                    "events": []}}, 
                         f, ensure_ascii=False, indent=2)
            continue
    
        total_events = len(all_events)

        if total_events == 0:
            print("    ⏩ File không có events. Ghi kết quả rỗng.")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({video_key: {"status": "empty", "events": []}},
                          f, ensure_ascii=False, indent=2)
            continue

        print(f"    📦 Có {total_events} events. Bắt đầu xử lý theo batch...")

        failed_batches = 0
        fixed_count = 0

        for i in range(0, total_events, BATCH_SIZE):
            batch = all_events[i:i + BATCH_SIZE]
            flat_events = [" ".join(e.get("text_lst", [])).strip() for e in batch]
            prompt = build_prompt(flat_events)

            fixed_texts = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    response_text = llm.prompt(prompt)
                    fixed_texts = parse_response(response_text, len(batch))
                    break
                except Exception as e:
                    print(f"    ⚠️ Lỗi cụm {i} (lần {attempt}/{MAX_RETRIES}): {e}")
                    if attempt < MAX_RETRIES:
                        backoff = attempt * 10.0
                        print(f"    💤 Nghỉ {backoff}s rồi thử lại...")
                        time.sleep(backoff)

            if fixed_texts is not None:
                for ev, txt in zip(batch, fixed_texts):
                    raw = " ".join(ev.get("text_lst", [])).strip()
                    ratio = len(txt) / max(len(raw),1)
                    if not 0.6 <= ratio <= 1.3:
                        print(f"    🚩 Tỷ lệ độ dài fixed/raw là {ratio:.2f}, cần xem lại")
                        print("============")
                        print(f"    - RAW: {raw}")
                        print("============")
                        print(f"    - FIXED: {txt}")
                        print("============")
                    print(f"Đã sửa: {raw[:20]}...{raw[-20:]} -> {txt[:20]}...{txt[-20:]}")
                    ev["text"] = txt
                    ev["correct_status"] = "success"
                fixed_count += len(batch)
            else:
                failed_batches += 1
                print(f"    ❌ Cụm {i} FAILED, để text rỗng.")
                for ev in batch:
                    ev["text"] = ""
                    ev["correct_status"] = "failed"
           
            if i + BATCH_SIZE < total_events:
                time.sleep(DELAY_BETWEEN_CALLS)

        status = "success" if failed_batches == 0 else ("partial" if fixed_count else "failed")
        
        slim_events = [
            {
                "text_lst": ev["text_lst"],
                "text": ev["text"],
                "correct_status": ev["correct_status"],
            }
            for ev in all_events
        ]
        
        output_data = {video_key: {
            "correct_status": status, 
            "group_status": info.get("status", "unknown"),
            "events": slim_events}}
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        print(f"    ✅ {filename}: status={status}, sửa {fixed_count}/{total_events} events -> {output_path}")

    print(f"\n⚔️ HOÀN THÀNH: LEVEL {LEVEL_NUMBER} ({group.upper()}) - PART {PART}/{TOTAL_PARTS}!")
    kbench.assertions.assert_true(True, expectation=f"Done group {group} Part {PART}")

In [ ]:
# Run the task with the specified LLM
correct_event_text_task.run(llm=llm)